In [ ]:
import xarray as xr
from sentinel_tiles import sentinel_tiles
import pyproj
import numpy as np
from matplotlib import pyplot as plt
import shapely
from functools import reduce

In [ ]:
detections = xr.open_dataset("detections.nc")
detections

In [ ]:
transformer = pyproj.Transformer.from_crs("EPSG:5071", "EPSG:4326", always_xy=True)

detection_footprints = [
    shapely.geometry.box(
        *transformer.transform_bounds(xmin, ymin, xmax, ymax)
    )
    for (xmin, ymin, xmax, ymax, tmin) in 
    zip(
     detections.xmin, detections.ymin, detections.xmax, detections.ymax, detections.tmin
    )
    if tmin > 2018
]

In [ ]:
detection_tilesets = list(map(sentinel_tiles.tiles, detection_footprints))

In [ ]:
print(len(detection_tilesets))

How many tiles per detection?

In [ ]:
n_tiles = np.array(list(map(len, detection_tilesets)))
plt.hist(n_tiles)

In [ ]:
all_tiles = reduce(set.union, detection_tilesets)
print(len(all_tiles))

In [ ]:
detection_tilesets_hashable = [
    "_".join(sorted(list(s))) for s in detection_tilesets
]

unique_tilesets = set(detection_tilesets_hashable)
print(len(unique_tilesets))

In [ ]:
detection_tilesets.sort(key=len, reverse=True)

In [ ]:
print(len(detection_tilesets[0]))

In [ ]:
detection_tilesets_filtered = []
for tileset in detection_tilesets:
    if not any(s.issuperset(tileset) for s in detection_tilesets_filtered):
        detection_tilesets_filtered.append(tileset)

In [ ]:
detection_tilesets_filtered

In [ ]:
len(detection_tilesets_filtered)